# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**One row means:** one content item for one pseudonymized client on one reporting date.

**Table:** `fact_content_daily_performance`.

**Time window:** March 2026 (`month=2026-03`). I use this mid-panel month as the development window rather than the final June 2026 sample.

**What I would predict/rank:** the eventual Search Intelligence task will use an observed declining-content outcome as the label/proxy and use the resulting score to prioritize content items for review.

**Deliberate exclusion:** `client_hash_id` and `content_hash_id` are identifiers used for grouping, joining, or splitting, not predictive features. Outcome-derived fields such as `trend_direction` and `trend_pct` are also excluded because they can leak outcome information into the model.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Access the Hugging Face token from Colab Secrets.
# Create a Colab Secret named HF_TOKEN and enable notebook access.
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access.")

print("HF_TOKEN loaded:", bool(HF_TOKEN))

HF_TOKEN loaded: True


In [2]:
# Connect to the Hugging Face warehouse with DuckDB.
# The token is passed as a session variable, not written into the SQL text.
import duckdb

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MAR = f"{FACT}/month=2026-03/*.parquet"

print("DuckDB connected.")
print("Working table: fact_content_daily_performance")
print("Working partition: March 2026")

DuckDB connected.
Working table: fact_content_daily_performance
Working partition: March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

2. Fields: feature / label / context / excluded
Candidate features (maximum of five)
- gsc_impressions — observed search impressions. Candidate feature; check the source availability flag and reporting-time caveat.
- gsc_clicks — observed search clicks. Candidate feature; check the source availability flag and reporting-time caveat.
- gsc_avg_position — observed average search position. A value of 0 may indicate no position data, so it should not automatically be interpreted as a literal rank.
- ga4_sessions — observed GA4 sessions, usable only when the relevant GA4 data is available.
- scroll_events — observed scroll-event count, usable only when the corresponding source data is available.
Label / proxy
The intended target is an observed future declining-content outcome. The March fact table does not directly expose is_declining_label, trend_direction, or trend_pct; therefore, the final decline label cannot be claimed or evaluated from this table alone.
For the leakage demonstration only, a temporary observed_target is defined as 1 when gsc_clicks > 0 and 0 otherwise. This is a demonstration target, not the final decline label.
Context
- report_date and month — identify the reporting time window.
- client_hash_id — pseudonymized client identifier for context, joins, grouping, or client-aware splitting.
- content_hash_id — pseudonymized content identifier for context or joins.
- gsc_data_available and ga4_data_available — indicate whether the corresponding source data is available.
Excluded and why
- client_hash_id and content_hash_id — identifiers, not generalizable performance measurements.
- Any future-outcome field or outcome-derived field — unavailable at the decision moment and may leak the answer.
- trend_direction and trend_pct — excluded from model features because dataset guidance describes them as outcome-derived; they are not present in the inspected March fact-table schema.
- ga4_sessions or scroll_events values without considering source availability — a missing/unavailable source must not be silently interpreted as a measured zero.

3. Verify it with queries, then build the feature frame and leakage trap
The next three verification queries check the stated grain, row count/date span, and GA4 availability. Their outputs should remain visible after running the notebook.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

3. Verify it with queries, then build the feature frame and leakage trap
The next three verification queries check the stated grain, row count/date span, and GA4 availability. Their outputs should remain visible after running the notebook.

Verification 1 — Does the observed data match the stated grain?
The proposed grain is one row per report_date × client_hash_id × content_hash_id. The query returns duplicate keys if any are found; zero rows supports the stated grain for this partition.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{MAR}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")
grain_check

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

Verification 2 — How many rows are in the March slice, and what dates does it cover?

In [4]:
date_span_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{MAR}')
""")
date_span_check

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘

Verification 3 — How many rows have GA4 data available?
IS TRUE explicitly counts only rows where the availability flag is true. IS NOT TRUE includes both false and null flags, so an unknown/null flag is not mistaken for available data.

In [5]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(ga4_data_available IS TRUE) AS available_rows,
    COUNTIF(ga4_data_available IS NOT TRUE) AS unavailable_or_not_true_rows
FROM read_parquet('{MAR}')
""")
availability_check

┌────────────┬────────────────┬──────────────────────────────┐
│ total_rows │ available_rows │ unavailable_or_not_true_rows │
│   int64    │     int128     │            int128            │
├────────────┼────────────────┼──────────────────────────────┤
│    9841378 │         413966 │                      9427412 │
└────────────┴────────────────┴──────────────────────────────┘

Five-feature frame
Cell 13 of 22
Five-feature frame
These five candidate features come from the same March 2026 partition. The query shows feature values alongside the date, identifiers, and source-availability flags so the data can be interpreted in context.

In [6]:
feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events,
    gsc_data_available,
    ga4_data_available
FROM read_parquet('{MAR}')
LIMIT 10
""")
feature_frame

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬───────────────────┬──────────────┬───────────────┬────────────────────┬────────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ ga4_sessions │ scroll_events │ gsc_data_available │ ga4_data_available │
│    date     │         varchar         │         varchar          │      int64      │   int64    │      double       │    int64     │     int64     │      boolean       │      boolean       │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼───────────────────┼──────────────┼───────────────┼────────────────────┼────────────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              20 │          0 │              3.35 │         NULL │          NULL │ true               │ NULL               │
│ 2026-03-01  │ client_73cda7b4e4f2

available when? — one line per candidate feature
- gsc_impressions: knowable at the decision moment only if the reporting snapshot has been published before the decision; check gsc_data_available.
- gsc_clicks: knowable at the decision moment only if the reporting snapshot has been published before the decision; check gsc_data_available.
- gsc_avg_position: knowable at the decision moment only if the reporting snapshot has been published before the decision; check gsc_data_available, and treat zero-position values cautiously.
- ga4_sessions: knowable at the decision moment only when GA4 data is available and the reporting snapshot has arrived; check ga4_data_available.
- scroll_events: knowable at the decision moment only when the corresponding analytics data is available and the reporting snapshot has arrived; check the applicable availability information.
Temporal rule: for a real future-decline model, the feature window must end before the outcome window begins. These fields are candidates, not yet approved final predictors, until that timing is defined.

The deliberate leakage trap
The following experiment uses real rows from the March warehouse. For this demonstration only, observed_target is defined as whether gsc_clicks > 0. The honest baseline uses gsc_impressions and gsc_avg_position; the deliberately leaked version adds target_leak, which copies the target itself. This is not the final decline model or its performance estimate.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Demonstration target only — not the final declining-content label.
leak_data = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_avg_position,
    CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END AS observed_target
FROM read_parquet('{MAR}')
WHERE gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
LIMIT 100000
""").df()

print("Leakage demonstration rows:", len(leak_data))
print("Demonstration target prevalence:", round(leak_data["observed_target"].mean(), 4))
leak_data.head()

Leakage demonstration rows: 100000
Demonstration target prevalence: 0.0568


,gsc_impressions,gsc_avg_position,observed_target
0,20,3.350000,0
1,1,0.000000,0
2,125,4.928000,1
3,7,4.000000,0
4,11,2.272727,0


In [8]:
# Honest baseline: no target-derived feature is included.
model_data = leak_data[
    ["gsc_impressions", "gsc_avg_position", "observed_target"]
].copy()

X = model_data[["gsc_impressions", "gsc_avg_position"]]
y = model_data["observed_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)
honest_pred = honest_model.predict(X_test)
honest_accuracy = accuracy_score(y_test, honest_pred)

print("Honest baseline accuracy:", round(honest_accuracy, 4))

Honest baseline accuracy: 0.9516


In [9]:
# Deliberate leak: copy the answer into a feature.
leaky_data = model_data.copy()
leaky_data["target_leak"] = leaky_data["observed_target"]

X_leaky = leaky_data[
    ["gsc_impressions", "gsc_avg_position", "target_leak"]
]
y = leaky_data["observed_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.20, random_state=42, stratify=y
)

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_model.fit(X_train, y_train)
leaky_pred = leaky_model.predict(X_test)
leaky_accuracy = accuracy_score(y_test, leaky_pred)

print("Accuracy with deliberate leakage:", round(leaky_accuracy, 4))

Accuracy with deliberate leakage: 1.0


In [10]:
# Remove the leaked column and rerun the same honest feature set.
clean_data = leaky_data.drop(columns=["target_leak"])
X_clean = clean_data[["gsc_impressions", "gsc_avg_position"]]
y = clean_data["observed_target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=0.20, random_state=42, stratify=y
)

clean_model = DecisionTreeClassifier(max_depth=3, random_state=42)
clean_model.fit(X_train, y_train)
clean_pred = clean_model.predict(X_test)
clean_accuracy = accuracy_score(y_test, clean_pred)

print("Honest accuracy after removing leakage:", round(clean_accuracy, 4))

assert leaky_accuracy >= honest_accuracy, "Expected the deliberately leaked score to be at least as high as baseline."
assert abs(clean_accuracy - honest_accuracy) < 1e-12, "Removing the leaked column should reproduce the honest baseline score."

Honest accuracy after removing leakage: 0.9516


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

4. Data limits
Named limitation — uneven analytics availability and missing final decline label.
In the March 2026 partition, the observed GA4 availability check previously returned 413,966 rows with ga4_data_available IS TRUE out of 9,841,378 total rows; 9,427,412 rows were false or null for that flag. Re-run Verification 3 to confirm these counts in this notebook. GA4-derived values cannot be assumed to be equally available across the slice.
Also, the inspected March fact-table schema does not directly contain the final declining-content label (is_declining_label, trend_direction, or trend_pct). The leakage experiment is only a controlled demonstration using a temporary clicks-based target; it does not establish performance for the intended decline-ranking task.
Before building the final model, define an observed future outcome, ensure features are available before the decision moment, check each source's availability, and keep the final June 2026 month sealed for testing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.